<a href="https://colab.research.google.com/github/elinimuleg00-bot/30-Days-of-Python-DevOps/blob/main/Day12/Day12_gpu_memory_checker.py" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# System Resource & GPU Telemetry Monitor
An MLOps hardware monitoring script demonstrating system command execution via Python's subprocess.run(), RAM parse logic using free -m, NVIDIA GPU telemetry extraction using nvidia-smi CLI query flags, fallback driver exception handling, and structured JSON telemetry export for observability pipelines.

In [ ]:
import json
import subprocess

def get_system_ram_mb():
  """Queries Linux system RAM metrics in megabytes using 'free -m'."""
  try:
    # Run free -m to get memory statistics in Megabytes
    # check=True: Forces subprocess to raise a CalledProcessError if the command fails, preventing silent pipeline failures.
    result = subprocess.run(["free","-m"], capture_output=True, text=True, check=True)

    line = result.stdout.strip().split("\n") #Cleans trailing whitespace and splits the terminal text into individual rows.
    mem_row = line[1].split()
    # line 0 is header, line 1 is Mem. Splits the Mem: line into individual data columns: Total, Used, & Available RAM

    return {
        "ram_total_mb": int(mem_row[1]),
        "ram_used_mb": int(mem_row[2]),
        "ram_available_mb": int(mem_row[6])
    }
  except Exception as e:
    return{"ram_error": str(e)}

def get_gpu_metrics():
  """Queries NVIDIA GPU utilization and VRAM using 'nvidia-smi'."""
  try:
    # Query nvidia-smi with CSV output format
    cmd=[
        "nvidia-smi",
        "--query-gpu=name,memory.total,memory.used,memory.free,utilization.gpu",
        # Instructs nvidia-smi to extract specific telemetry metrics instead of printing the full terminal graphic.
        "--format=csv,noheader,nounits"
        #Strips header text and unit labels (e.g., returning 15360 instead of 15360 MiB) so Python can convert values directly to integers.
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)

    # Parse CSV line: Name, Total VRAM, Used VRAM, Free VRAM, GPU Util %
    data = result.stdout.strip().split(", ") # since its already a single row foratted with commas, we do not need to split lines or navigate rows, so breaking it by comma is enough
    return {
       "gpu_name": data[0],
       "vram_total_mb": int(data[1]),
       "vram_used_mb": int(data[2]),
       "vram_free_mb": int(data[3]),
       "gpu_utilization_pct": int(data[4]),
    }
  except FileNotFoundError:
    return{ # Because the session is CPU-only, the nvidia-smi driver executable does not exist on disk.
       "gpu_status": "No NVIDIA GPU detected (CPU-only execution enviroment)"
    }
  except Exception as e:
      return {"gpu_error": str(e)}

def generate_resource_report():
  """Combines system RAM and GPU metrics into an MLOps-ready log payload."""

  report = {
       "system_ram": get_system_ram_mb(),
       "gpu_hardware": get_gpu_metrics()
   }

  print(" --- MLOps Resource Utilization Report --- ")
  print(json.dumps(report, indent = 4)) # Formats the dictionary into a clean JSON string for readable terminal logging.

  # Export for MLOps experiment tracking
  # we need the .json file to record how much memory was used during every run
  output_path = "system_resource_metrics.json"
  with open(output_path, "w") as f:
    json.dump(report, f, indent=4) # Writes the resource telemetry out to system_resource_metrics.json for consumption by monitoring tools like Prometheus, Weights & Biases, or MLflow.
  print(f"\n Resource metrics exported to '{output_path}")

if __name__ == "__main__":
  generate_resource_report()




 --- MLOps Resource Utilization Report --- 
{
    "system_ram": {
        "ram_total_mb": 12975,
        "ram_used_mb": 834,
        "ram_available_mb": 11858
    },
    "gpu_hardware": {
        "gpu_status": "No NVIDIA GPU detected (CPU-only execution enviroment)"
    }
}

 Resource metrics exported to 'system_resource_metrics.json
